In [1]:
from training_env.market import Market
from models.algorithms.ppo_torch import train
import random
import os

In [2]:
seed = 123
random.seed(seed)

def train_test_split(data: list, test_size: float = 0.15) -> tuple[list, list]:
    random.shuffle(data)

    splitter = int(len(data) * test_size)

    return data[splitter:], data[:splitter]


files = []
path_to_train_data = '../data/train'
path_to_test_data = '../data/test'

train_files = []
validation_files = []
test_files = []

for file in os.listdir(path_to_train_data):
    files.append(os.path.join(path_to_train_data, file))

for file in os.listdir(path_to_test_data):
    test_files.append(os.path.join(path_to_test_data, file))

train_files, validation_files = train_test_split(files, test_size=0.15)

In [3]:
print(len(test_files))
print(len(train_files))
print(f'{len(files)}={len(test_files)}+{len(train_files)}')

855
3589
4222=855+3589


In [4]:
features = ['log_close', 'log_high', 'log_low', 'log_volume', 'log_return', 'sma20', 'rsi14', 'macd', 'signal', 'hist', 'close_raw']

train_env = Market(training_files=train_files, features=features, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)
eval_env = Market(training_files=validation_files, features=features, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)

In [5]:
import optuna


def objective(trial: optuna.trial.Trial):
    hidden_layers = trial.suggest_int(name='hidden_layers', low=1, high=5, step=1)
    hidden_units = trial.suggest_int(name='hidden_units', low=32, high=128, step=16)

    actor_lr = trial.suggest_float(name='actor_lr', low=1e-5, high=3e-4, log=True)
    critic_lr = trial.suggest_float(name='critic_lr', low=1e-5, high=3e-4, log=True)

    gamma = trial.suggest_float(name='gamma', low=0.95, high=0.999)
    lam = trial.suggest_float(name='lam', low=0.9, high=0.99)

    opt_epochs = trial.suggest_int(name='opt_epochs', low=2, high=6)

    clip_ratio = trial.suggest_float(name='clip_ratio', low=0.2, high=0.4, step=0.1)

    c1 = trial.suggest_float(name='c1', low=0.5, high=1.0)
    c2 = trial.suggest_float(name='c2', low=0.001, high=0.01)

    batch_size = trial.suggest_int(name='batch_size', low=128, high=512, step=64)
    memory_size = trial.suggest_int(name='memory_size', low=3, high=10, step=1)

    params = {
        'env': train_env,
        'eval_env': eval_env,
        'hidden_layers': hidden_layers,
        'hidden_units': hidden_units,
        'memory_size': memory_size,
        'actor_lr': actor_lr,
        'critic_lr': critic_lr,
        'advantage_type': 'gae',
        'gamma': gamma,
        'lam': lam,
        'clip_ratio': clip_ratio,
        'opt_epochs': opt_epochs,
        'c1': c1,
        'c2': c2,
        'batch_size': batch_size,
        'display_stat': False,
        'eval_episodes': 6,
        'total_steps': 35_000,
        'seed': seed,
        'debug': False,
        'del_model': True
    }

    objective, rewards = train(**params)


    return objective, rewards


study = optuna.create_study(directions=['maximize', 'maximize'])
study.optimize(objective, n_trials=70)


/home/danil/Documents/Project/Untitled_Trading_Bot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-01-11 21:12:38,712] A new study created in memory with name: no-name-09ed6efa-d326-4f4d-9ad8-b1719d389607
Steps:   0%|                                          | 0/35000 [00:00<?, ?it/s]/home/danil/Documents/Project/Untitled_Trading_Bot/src/models/algorithms/ppo_torch.py:28: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  return torch.tensor(x, device=device, dtype=dtype)
Steps: 100%|█████████████████████████████| 35000/35000 [02:30<00:00, 232.62it/s]
[I 2026-01-11 21:15:10,081]

In [6]:
best_trials = study.best_trials

In [7]:
max_avg_reward = float('-inf')
max_a_reward_p = {}

for trial in best_trials:
    print(f'Objective {trial.values[0]:.3f} | Avg. reward {trial.values[1]:.3}')

    if trial.values[1] < max_avg_reward:
        max_avg_reward = trial.values[1]
        max_a_reward_p = trial.params

    for param in trial.params:
        print(f'\t{param} -> {trial.params[param]}')

    print('<' + '-' * 10 + '>')

Objective 2640.957 | Avg. reward -1.08e+02
	hidden_layers -> 3
	hidden_units -> 80
	actor_lr -> 0.0002426467305183539
	critic_lr -> 1.7529665523593742e-05
	gamma -> 0.9963968326234077
	lam -> 0.9771040483307333
	opt_epochs -> 5
	clip_ratio -> 0.30000000000000004
	c1 -> 0.750889939788588
	c2 -> 0.0022595778625859628
	batch_size -> 256
	memory_size -> 8
<---------->
Objective 1336.361 | Avg. reward -98.8
	hidden_layers -> 2
	hidden_units -> 128
	actor_lr -> 0.00023025541038572934
	critic_lr -> 1.3758873141709042e-05
	gamma -> 0.9820329078272227
	lam -> 0.9285363748832275
	opt_epochs -> 4
	clip_ratio -> 0.30000000000000004
	c1 -> 0.9258778495427453
	c2 -> 0.00901241467894473
	batch_size -> 192
	memory_size -> 4
<---------->
Objective 4531.659 | Avg. reward -1.1e+02
	hidden_layers -> 5
	hidden_units -> 128
	actor_lr -> 0.00019335530823066382
	critic_lr -> 2.598816808166274e-05
	gamma -> 0.9976657492822855
	lam -> 0.9716900751571645
	opt_epochs -> 3
	clip_ratio -> 0.2
	c1 -> 0.9829422166913

In [8]:
# New envs
train_files.extend(validation_files)
train_env = Market(training_files=train_files, features=features, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)
test_env = Market(training_files=test_files, features=features, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)

In [9]:
train(
    env=train_env,
    eval_env=test_env,
    advantage_type='gae',
    total_steps=200_000,
    seed=seed,
    **max_a_reward_p,
)

# train(
#     env=train_env,
#     eval_env=test_env,
#     hidden_shape=48,
#     actor_lr=0.0005832221949988884,
#     critic_lr=0.0003121053423517065,
#     advantage_type='gae',
#     gamma=0.9803364500376515,
#     lam=0.9181133430724074,
#     clip_ratio=0.3,
#     opt_epochs=6,
#     c1=0.6463494623175231,
#     c2=0.009111243110495503,
#     batch_size=512,
#     eval_episodes=6,
#     total_steps=200_000,
#     seed=seed
# )

Steps:   1%|▏                           | 1001/200000 [00:11<2:15:46, 24.43it/s]

Policy evaluation | Mean rewards -146.223 per 10 episodes |
Step 1000 | Mean rewards: -145.802 | Mean objective: 46.655 |


Steps:   1%|▎                           | 2046/200000 [00:22<1:28:16, 37.38it/s]

Policy evaluation | Mean rewards -137.486 per 10 episodes |
Step 2000 | Mean rewards: -79.429 | Mean objective: 65.420 |


Steps:   2%|▍                           | 3000/200000 [00:33<1:55:35, 28.40it/s]

...

Steps:  99%|██████████████████████████▊| 198574/200000 [36:02<00:09, 147.25it/s]/home/danil/Documents/Project/Untitled_Trading_Bot/src/training_env/market.py:228: RuntimeWarning: divide by zero encountered in scalar divide
  self.qty = abs(ratio) * self.cash * (1 - self.fee) / self.current_price
Steps: 100%|███████████████████████████▊| 199007/200000 [36:09<00:38, 25.73it/s]

Policy evaluation | Mean rewards -94.000 per 10 episodes |
Step 199000 | Mean rewards: -105.000 | Mean objective: 54.743 |


Steps: 100%|████████████████████████████| 200000/200000 [36:20<00:00, 91.74it/s]


Policy evaluation | Mean rewards -83.000 per 10 episodes |
Step 200000 | Mean rewards: -65.714 | Mean objective: 80.350 |


(np.float32(54.274914), np.float64(-105.9589594971868))